## weights-tag-parents.ipynb

Builds a sparse parent-genre feature matrix from the cleaned `tag_parents.csv`
produced by `tag-hierarchy.ipynb`.

### Logic
For each album, for each parent genre:
- Find all child tags of that parent that the album has
- Take the **max child tag weight** as the parent column value

Max (not sum) is used so a highly-tagged metal album doesn't score higher than
a lightly-tagged one just because it has more metal subgenre tags.

### Output
| File | Description |
|---|---|
| `data/features/album_tag_parent_matrix.npz` | Sparse matrix (n_albums × n_parents), aligned to `album_ids.pkl` |

> **Optional under Option A (2026-06-10).** `02-feature-genre.ipynb` now folds the coarse
> parent rollup directly into `album_genre_matrix.npz` (one Genre knob scales fine + coarse),
> so the standalone `album_tag_parent_matrix.npz` built here is no longer required by the app.
> Keep this notebook only for standalone analysis of the 20-column parent matrix.

In [1]:
import pandas as pd
import numpy as np
import pickle
from scipy.sparse import csr_matrix, save_npz
from collections import defaultdict

DATA_DIR     = '../data'
FEATURES_DIR = '../data/features'

### 1. Load inputs

In [10]:
# Master album row index — alignment contract for all feature matrices
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = np.array(pickle.load(f))
n_albums = len(album_ids)
album_id_to_row = {aid: i for i, aid in enumerate(album_ids)}
print(f'Albums in index: {n_albums:,}')

# Raw album tags
album_tags = pd.read_parquet(f'{DATA_DIR}/mb_album_tag.parquet')
print(f'Album-tag rows : {len(album_tags):,}')

# Tag id → name
tags = pd.read_parquet(f'{DATA_DIR}/mb_tag.parquet')
tags['name'] = tags['name'].str.strip().str.lower()
tag_id_to_name = tags.set_index('id')['name'].to_dict()
tag_name_to_id = tags.set_index('name')['id'].to_dict()

# Cleaned parent map
parent_map = pd.read_csv(f'{DATA_DIR}/tag_parents.csv')
print(f'Child→parent pairs: {len(parent_map):,}')
print(f'Unique parents     : {parent_map["parent"].nunique()}')

Albums in index: 1,008,102
Album-tag rows : 3,042,637
Child→parent pairs: 439
Unique parents     : 20


### 2. Compute normalised child tag weights

Use the same normalisation as the existing tag matrix:
weight = tag_count / max_tag_count_for_that_album (per-album max normalisation).

In [11]:
# Per-album max tag count for normalisation
album_max = album_tags.groupby('album_id')['tag_count'].max().rename('max_count')
album_tags = album_tags.join(album_max, on='album_id')
album_tags['weight'] = album_tags['tag_count'] / album_tags['max_count']

# Map tag_id → tag_name
album_tags['tag_name'] = album_tags['tag_id'].map(tag_id_to_name)

# Only keep rows whose tag has a parent mapping
child_to_parent = parent_map.set_index('child')['parent'].to_dict()
album_tags['parent'] = album_tags['tag_name'].map(child_to_parent)
tagged = album_tags.dropna(subset=['parent']).copy()

print(f'Album-tag rows with a parent: {len(tagged):,}')
print(f'Albums covered               : {tagged["album_id"].nunique():,}')

Album-tag rows with a parent: 571,163
Albums covered               : 378,446


### 3. Aggregate to parent level (max child weight per album × parent)

In [12]:
# For each (album, parent) pair take the max weight across all child tags
agg = (
    tagged
    .groupby(['album_id', 'parent'])['weight']
    .max()
    .reset_index()
    .rename(columns={'weight': 'parent_weight'})
)

print(f'(album, parent) pairs: {len(agg):,}')
print(f'\nMean parent weight : {agg["parent_weight"].mean():.4f}')
print(f'Median parent weight: {agg["parent_weight"].median():.4f}')

# Distribution across parents
print('\nAlbums per parent genre:')
print(
    agg.groupby('parent')['album_id']
    .nunique()
    .sort_values(ascending=False)
    .to_string()
)

(album, parent) pairs: 487,827

Mean parent weight : 0.9330
Median parent weight: 1.0000

Albums per parent genre:
parent
rock            159629
metal            61743
indie            45731
house            24202
psychedelic      20470
pop              20406
jazz             20354
ambient          17373
blues            17239
punk             15290
progressive      13709
hardcore         11488
country          10512
classical        10226
trance            9646
alternative       6896
experimental      6089
dance             5967
noise             5440
instrumental      5417


### 4. Build sparse matrix aligned to album_ids.pkl

In [13]:
# Parent genre columns — deterministic ordering
parents_sorted = sorted(agg['parent'].unique())
parent_to_col  = {p: i for i, p in enumerate(parents_sorted)}
n_parents      = len(parents_sorted)
print(f'Parent columns: {n_parents}')
print(f'Parents: {parents_sorted}')

# Map album_id → row index, drop albums not in index
agg['row'] = agg['album_id'].map(album_id_to_row)
agg['col'] = agg['parent'].map(parent_to_col)
agg = agg.dropna(subset=['row'])
agg['row'] = agg['row'].astype(int)

# Build CSR
X_parent = csr_matrix(
    (agg['parent_weight'].values, (agg['row'].values, agg['col'].values)),
    shape=(n_albums, n_parents),
    dtype=np.float32,
)

print(f'\nMatrix shape : {X_parent.shape}')
print(f'Non-zero     : {X_parent.nnz:,}')
print(f'Density      : {X_parent.nnz / (X_parent.shape[0] * X_parent.shape[1]):.4%}')
print(f'Albums with ≥1 parent signal: {(X_parent.sum(axis=1) > 0).sum():,}')

Parent columns: 20
Parents: ['alternative', 'ambient', 'blues', 'classical', 'country', 'dance', 'experimental', 'hardcore', 'house', 'indie', 'instrumental', 'jazz', 'metal', 'noise', 'pop', 'progressive', 'psychedelic', 'punk', 'rock', 'trance']

Matrix shape : (1008102, 20)
Non-zero     : 487,827
Density      : 2.4195%
Albums with ≥1 parent signal: 378,446


### 5. Sanity checks

In [14]:
# Check a known metal album has a high metal score
# (replace with any album_id you know is metal)
metal_col = parent_to_col.get('metal')
rock_col  = parent_to_col.get('rock')

# Top 5 albums by metal score
metal_scores = np.array(X_parent[:, metal_col].todense()).ravel()
top_metal_rows = np.argsort(metal_scores)[::-1][:5]
print('=== Top 5 albums by metal parent score ===')
for r in top_metal_rows:
    print(f'  album_id={album_ids[r]}  score={metal_scores[r]:.4f}')

# Top 5 albums by rock score
rock_scores = np.array(X_parent[:, rock_col].todense()).ravel()
top_rock_rows = np.argsort(rock_scores)[::-1][:5]
print('\n=== Top 5 albums by rock parent score ===')
for r in top_rock_rows:
    print(f'  album_id={album_ids[r]}  score={rock_scores[r]:.4f}')

# Verify row alignment: shape[0] must equal len(album_ids)
assert X_parent.shape[0] == n_albums, 'Row count mismatch!'
print(f'\nAlignment check passed: {X_parent.shape[0]:,} rows == {n_albums:,} album_ids')

=== Top 5 albums by metal parent score ===
  album_id=3052860  score=1.0000
  album_id=1974203  score=1.0000
  album_id=1987187  score=1.0000
  album_id=108989  score=1.0000
  album_id=2085560  score=1.0000

=== Top 5 albums by rock parent score ===
  album_id=4778806  score=1.0000
  album_id=30  score=1.0000
  album_id=4776437  score=1.0000
  album_id=2879103  score=1.0000
  album_id=2879085  score=1.0000

Alignment check passed: 1,008,102 rows == 1,008,102 album_ids


### 6. Save

In [15]:
import os, json

out_path = f'{FEATURES_DIR}/album_tag_parent_matrix.npz'
save_npz(out_path, X_parent)
print(f'Saved → {out_path}')

# Save the column index so the model rebuild cell can inspect it
col_index_path = f'{FEATURES_DIR}/album_tag_parent_columns.json'
with open(col_index_path, 'w') as f:
    json.dump(parents_sorted, f, indent=2)
print(f'Saved column index → {col_index_path}')

print(f'\nFinal matrix: {X_parent.shape[0]:,} albums × {X_parent.shape[1]} parent genres')

Saved → ../data/features/album_tag_parent_matrix.npz
Saved column index → ../data/features/album_tag_parent_columns.json

Final matrix: 1,008,102 albums × 20 parent genres


In [8]:
import pandas as pd

DATA_DIR = '../data'
parent_map = pd.read_csv(f'{DATA_DIR}/tag_parents.csv')

# Check if any of our intended top-level parents appear as children too
top_parents = {'rock', 'metal', 'indie', 'house', 'jazz', 'pop', 'psychedelic',
               'blues', 'ambient', 'punk', 'progressive', 'hardcore', 'country',
               'trance', 'classical', 'alternative', 'dance', 'experimental',
               'noise', 'instrumental'}

# Parents that are also children somewhere
also_children = parent_map[parent_map['child'].isin(top_parents)]
print(f'Top-level parents that are also mapped as children: {len(also_children)}')
print(also_children[['child', 'parent', 'child_albums']].to_string(index=False))

# How many unique parents are there total?
print(f'\nTotal unique parents in map: {parent_map["parent"].nunique()}')
print(parent_map['parent'].value_counts().head(30))

Top-level parents that are also mapped as children: 0
Empty DataFrame
Columns: [child, parent, child_albums]
Index: []

Total unique parents in map: 403
parent
rock            64
cover           50
metal           42
pop             38
country         35
blues           27
jazz            27
punk            23
alternative     23
classical       22
house           20
dance           20
soul            20
folk            18
rap             18
hardcore        17
ambient         16
trance          13
hip hop         13
disco           13
funk            13
instrumental    12
black metal     12
industrial      12
jazzthing       12
hip-hop         12
experimental    11
noise           10
beat            10
electronic      10
Name: count, dtype: int64


In [9]:
import pandas as pd

DATA_DIR = '../data'
parent_map = pd.read_csv(f'{DATA_DIR}/tag_parents.csv')

VALID_PARENTS = {
    'rock', 'metal', 'indie', 'house', 'jazz', 'pop', 'psychedelic',
    'blues', 'ambient', 'punk', 'progressive', 'hardcore', 'country',
    'trance', 'classical', 'alternative', 'dance', 'experimental',
    'noise', 'instrumental'
}

before = len(parent_map)
parent_map = parent_map[parent_map['parent'].isin(VALID_PARENTS)]
after = len(parent_map)

print(f'Rows before : {before:,}')
print(f'Rows after  : {after:,}')
print(f'Dropped     : {before - after:,}')
print(f'\nRemaining parents ({parent_map["parent"].nunique()}):')
print(parent_map.groupby('parent')['child_albums'].sum().sort_values(ascending=False))

parent_map.to_csv(f'{DATA_DIR}/tag_parents.csv', index=False)
print('\nSaved cleaned tag_parents.csv')

Rows before : 1,366
Rows after  : 439
Dropped     : 927

Remaining parents (20):
parent
rock            202276
metal            80438
indie            51006
house            28613
jazz             22358
pop              21173
psychedelic      21001
blues            18999
ambient          18023
punk             16276
progressive      14858
hardcore         12332
country          11420
trance           10815
classical        10521
alternative       7343
dance             6266
experimental      6223
noise             5657
instrumental      5565
Name: child_albums, dtype: int64

Saved cleaned tag_parents.csv


In [16]:
import pandas as pd
import os

DATA_DIR = '../data'
lookup = (
    pd.read_parquet(f'{DATA_DIR}/mb_album_artists.parquet',
                    columns=['album_id', 'album_name', 'artist_name'])
    .drop_duplicates(subset='album_id')
    .set_index('album_id')
)

metal_ids = [3052860, 1974203, 1987187, 108989, 2085560]
rock_ids  = [4778806, 30, 4776437, 2879103, 2879085]

print('=== Top metal ===')
print(lookup.loc[lookup.index.isin(metal_ids), ['album_name', 'artist_name']])

print('\n=== Top rock ===')
print(lookup.loc[lookup.index.isin(rock_ids), ['album_name', 'artist_name']])

=== Top metal ===
                    album_name   artist_name
album_id                                    
108989            Witch Hunter  Grave Digger
1974203              Breathing      Niboowin
1987187        I Shall Conquer     Leviticus
2085560        Dark Revelation    Buried God
3052860   Низведанье: Глава 26    Дьяволиада

=== Top rock ===
                                                 album_name  \
album_id                                                      
30                                                Let It Be   
2879085   Raiders of the Lost Archives: Demos & Rarities...   
2879103                                           In Season   
4776437                         Forever Dead!y Demos (2017)   
4778806          Black Pearls, Vol. 5: Let's Rock And Roll!   

                 artist_name  
album_id                      
30          The Replacements  
2879085   Cult of Dom Keller  
2879103           White Duck  
4776437        Forever Dead!  
4778806              